# درس ۱۲ - کاهش تاریخچه چت با استفاده از دفترچه یادداشت Agent

این دفترچه یادداشت نشان می‌دهد چگونه می‌توان با استفاده از چارچوب Agent مایکروسافت، مدیریت زمینه را در گفتگوهای طولانی انجام داد. با افزایش طول گفتگوها، تعداد توکن‌ها افزایش می‌یابد — که در نهایت از پنجره زمینه مدل فراتر می‌رود. ما این مشکل را با **الگوی خلاصه‌سازی زمینه** و **دفترچه یادداشت agent** برای حافظه ماندگار حل می‌کنیم.

## آنچه یاد خواهید گرفت:
۱. **چرا مدیریت زمینه اهمیت دارد**: درک محدودیت‌های توکن و پنجره‌های زمینه  
۲. **عامل‌های آگاه به زمینه**: ساخت عامل‌هایی که زمینه گفتگوهای خود را مدیریت می‌کنند  
۳. **الگوی خلاصه‌سازی زمینه**: استفاده از ابزارها برای فشرده‌سازی تاریخچه گفتگو  
۴. **دفترچه یادداشت Agent**: حافظه ماندگاری که پس از کاهش زمینه باقی می‌ماند  

## پیش‌نیازها:
- راه‌اندازی Azure OpenAI با تنظیم متغیرهای محیطی  
- درک مفاهیم پایه agent از درس‌های قبلی


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## چرا مدیریت زمینه مهم است

هر مدل زبان بزرگ (LLM) دارای یک **پنجره زمینه‌ای** محدود است — حداکثر تعداد توکن‌هایی که می‌تواند در یک درخواست پردازش کند. با پیشرفت یک مکالمه چند مرحله‌ای:

- **تعداد توکن‌ها به صورت خطی افزایش می‌یابد** با هر پیام کاربر و پاسخ دستیار.
- **توکن‌های پرامپت هزینه را تسلط می‌دهند** زیرا کل تاریخچه در هر مرحله مجدداً ارسال می‌شود.
- در نهایت مکالمه **از پنجره زمینه‌ای فراتر می‌رود** و مدل یا برش می‌دهد یا خطا می‌دهد.

### استراتژی‌های مدیریت زمینه

| استراتژی | چگونه کار می‌کند | معاوضه |
|---|---|---|
| **برش** | پیام‌های قدیمی‌تر را حذف می‌کند | زمینه اولیه از دست می‌رود |
| **خلاصه‌سازی** | پیام‌های قدیمی‌تر را به یک خلاصه تبدیل می‌کند | برخی جزئیات از دست می‌روند، اما نکات کلیدی حفظ می‌شود |
| **دفترچه یادداشت / حافظه خارجی** | حقایق کلیدی را خارج از مکالمه ذخیره می‌کند | نیاز به فراخوانی ابزار دارد، اما در برابر هر کاهش مقاوم است |

در این دفترچه ما **خلاصه‌سازی** را با یک **ابزار دفترچه یادداشت** ترکیب می‌کنیم تا عامل بتواند پیوستگی را حفظ کند حتی زمانی که تاریخچه مکالمه خلاصه می‌شود.


## ایجاد یک عامل آگاه به زمینه


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## شبیه‌سازی یک مکالمه طولانی

بیایید یک مکالمه چندمرحله‌ای را مرور کنیم تا ببینیم چگونه زمینه جمع می‌شود. نماینده باید جزئیات کلیدی (ترجیحات، بودجه، تاریخ‌های سفر) را در طول گفتگو حفظ کند و پیوستگی را نشان دهد.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

توجه کنید که چگونه عامل زمینه را از نوبت‌های قبلی حفظ می‌کند — او درباره ژاپن، سوشی، معابد، عکاسی، بودجه ۳۰۰۰ دلاری، سفر انفرادی و بازه زمانی آوریل مطلع است. در یک مکالمه کوتاه این روش خوب عمل می‌کند، اما هر چه مکالمه طولانی‌تر شود، ارسال مجدد کل تاریخچه هزینه‌بر خواهد بود.

بیایید مکالمه را با نوبت‌های بیشتر ادامه دهیم تا تجمع زمینه را ببینیم:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## الگوی خلاصه‌سازی متن زمینه

با گسترش گفتگو، می‌توانیم از **ابزار خلاصه‌سازی** برای جمع‌وجور کردن متن تجمع‌یافته به فرمتی فشرده استفاده کنیم. عامل این ابزار را فراخوانی می‌کند تا ترجیحات کلیدی را ثبت کند تا حتی اگر پیام‌های قدیمی‌تر حذف شوند، اطلاعات حیاتی حفظ شود.

این الگو ساختار اصلی برای کاهش پیشرفته‌تر تاریخچه است:
1. عامل نکات کلیدی از گفتگو را شناسایی می‌کند
2. ابزار خلاصه‌سازی را فراخوانی می‌کند تا آن‌ها را ذخیره کند
3. پیام‌های قدیمی‌تر به‌طور ایمن حذف می‌شوند چون خلاصه آنچه اهمیت دارد را در بر می‌گیرد

در ادامه ما ابزاری به نام `summarize_preferences` تعریف می‌کنیم که عامل می‌تواند آن را برای ثبت خلاصه‌ای فشرده از آنچه یاد گرفته است، فراخوانی کند.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصه

در این درس یاد گرفتید چگونه در مکالمات طولانی‌مدت عامل‌ها، مدیریت زمینه را با استفاده از چارچوب عامل مایکروسافت انجام دهید:

### مفاهیم کلیدی
- **پنجره‌های زمینه محدود هستند** — هر توکن در تاریخچه مکالمه هزینه‌بر است و به محدودیت افزوده می‌شود.
- **ابزارهای خلاصه‌سازی** به عامل این امکان را می‌دهند که زمینه انباشته‌شده را در قالب خلاصه‌های فشرده جمع کند، استفاده از توکن‌ها را کاهش دهد و اطلاعات حیاتی را حفظ کند.
- **برگه یادداشت عامل** حافظه خارجی پایدار فراهم می‌کند که پس از هر کاهش مکالمه باقی می‌ماند.

### آنچه ساختید
- یک **عامل آگاه به زمینه** که تداوم را در مکالمات چندمرحله‌ای حفظ می‌کند
- یک **ابزار خلاصه‌سازی** (`summarize_preferences`) که جزئیات کلیدی کاربر را در قالب فشرده ضبط می‌کند
- یک **مکالمه چندمرحله‌ای** که حفظ زمینه و مدیریت تغییرات را نشان می‌دهد

### کاربردهای دنیای واقعی
- **ربات‌های خدمات مشتری**: ترجیحات را در جلسات طولانی پشتیبانی به خاطر می‌سپارند
- **دستیارهای شخصی**: پیگیری پروژه‌های جاری بدون نیاز به توضیح مجدد زمینه
- **معلمان آموزشی**: حفظ پیشرفت دانش‌آموز در تعاملات متعدد

### گام‌های بعدی
- پیاده‌سازی یک ابزار برگه یادداشت کامل با قابلیت پایدارسازی مبتنی بر فایل
- افزودن برش خودکار تاریخچه پس از خلاصه‌سازی
- ترکیب با پایگاه‌های داده برداری برای جستجوی حافظه معنایی
- ساخت عامل‌هایی که بتوانند روزها بعد با زمینه کامل به مکالمات ادامه دهند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح مهم**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما تلاش خود را برای دقت انجام داده‌ایم، لطفاً آگاه باشید که ترجمه‌های خودکار ممکن است دارای خطاها یا نواقصی باشند. سند اصلی به زبان مادری خود منبع معتبر محسوب می‌شود. برای اطلاعات حیاتی، ترجمه انسانی حرفه‌ای توصیه می‌شود. ما مسئول هیچ گونه سوتفاهم یا برداشت نادرست ناشی از استفاده این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
